In [1]:
# --------------------------------
# Task 5
# --------------------------------


import csv
import os
import logging
from functools import reduce

# --------------------------------
# Logging Configuration
# --------------------------------

logging.basicConfig(
    level=logging.INFO,
    format="%(levelname)s : %(message)s"
)

# --------------------------------
# Configuration
# --------------------------------

FILE_NAME = "orders.csv"

DISCOUNT_RULES = {
    "electronics": 0.10,
    "clothing": 0.20,
    "groceries": 0.05
}

MIN_ORDER_VALUE = 100


# --------------------------------
# Auto Generate CSV File
# --------------------------------

def create_csv_if_missing(file_name):

    if not os.path.exists(file_name):

        data = [
            ["id","category","price"],
            [1,"electronics",1000],
            [2,"clothing",200],
            [3,"electronics",500],
            [4,"groceries",50],
            [5,"clothing",300],
            [6,"groceries",120]
        ]

        with open(file_name,"w",newline="") as file:
            writer = csv.writer(file)
            writer.writerows(data)

        logging.info("CSV file auto-generated")


# --------------------------------
# Load Orders From CSV
# --------------------------------

def load_orders(file_name):

    try:
        with open(file_name,"r") as file:

            reader = csv.DictReader(file)

            orders = list(map(
                lambda row:{
                    "id": int(row["id"]),
                    "category": row["category"],
                    "price": float(row["price"])
                },
                reader
            ))

        logging.info("Orders loaded successfully")
        return orders

    except FileNotFoundError:

        logging.error("CSV file not found")
        return []


# --------------------------------
# Apply Discount
# --------------------------------

apply_discount = lambda order: {
    "id": order["id"],
    "category": order["category"],
    "original_price": order["price"],
    "final_price": round(
        order["price"] * (1 - DISCOUNT_RULES.get(order["category"],0)),
        2
    )
}


# --------------------------------
# Filter Valid Orders
# --------------------------------

filter_valid = lambda orders: list(
    filter(lambda o: o["final_price"] >= MIN_ORDER_VALUE, orders)
)


# --------------------------------
# Calculate Revenue
# --------------------------------

calculate_revenue = lambda orders: reduce(
    lambda total,o: total + o["final_price"],
    orders,
    0
)


# --------------------------------
# Tabular Output
# --------------------------------

def print_table(orders):

    print("\nProcessed Orders\n")

    print(f"{'ID':<5}{'Category':<15}{'Original':<15}{'Final':<10}")
    print("-"*45)

    list(map(lambda o:
        print(f"{o['id']:<5}{o['category']:<15}{o['original_price']:<15}{o['final_price']:<10}")
    , orders))


# --------------------------------
# Main Pipeline
# --------------------------------

def main():

    create_csv_if_missing(FILE_NAME)

    orders = load_orders(FILE_NAME)

    discounted_orders = list(map(apply_discount, orders))

    valid_orders = filter_valid(discounted_orders)

    revenue = calculate_revenue(valid_orders)

    print_table(valid_orders)

    print("\nTotal Orders:", len(valid_orders))
    print("Total Revenue:", revenue)


if __name__ == "__main__":
    main()

INFO : Orders loaded successfully



Processed Orders

ID   Category       Original       Final     
---------------------------------------------
1    electronics    1000.0         900.0     
2    clothing       200.0          160.0     
3    electronics    500.0          450.0     
5    clothing       300.0          240.0     
6    groceries      120.0          114.0     

Total Orders: 5
Total Revenue: 1864.0
